# Prepare Data for RDR: Version 1

The original dataset: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

## Setup

In [1]:
import numpy as np
import pandas as pd
import json

## Load Data

In [2]:
df = pd.read_csv('../data/raw/tmdb_5000_movies.csv')
df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [3]:
df.shape

(4803, 20)

## EDA
### Remove Columns

In [4]:
cols = df.columns
cols

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

Some columns are not important for simple movie recommendations because they are either identifiers, redundant information, unstructured text, or features that add unnecessary complexity for rule-based recommendation.

The following columns are removed:

- `homepage` - website link, not useful for recommendation
- `id` - unique identifier only
- `keywords` - unstructured text, harder for simple rule creation
- `original_title` - redundant since `title` is already available
- `overview` - long text description, not needed for basic rules
- `production_companies` - too specific and less useful for general recommendation
- `production_countries` - often redundant with language preference
- `spoken_languages` - overlaps with `original_language` and can be messy
- `status` - usually values like Released, not helpful
- `tagline` - short promotional text, not useful for recommendation

In [5]:
cols_to_remove = ['homepage', 'id', 'keywords', 'original_title', 'overview',
                'production_companies', 'production_countries', 'spoken_languages', 
                'status', 'tagline']

# romove those columns
df = df.drop(cols_to_remove, axis=1)
df.head()

,budget,genres,original_language,popularity,release_date,revenue,runtime,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,150.437577,2009-12-10,2787965087,162.0,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",en,139.082615,2007-05-19,961000000,169.0,Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,107.376788,2015-10-26,880674609,148.0,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",en,112.312950,2012-07-16,1084939099,165.0,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,43.926995,2012-03-07,284139100,132.0,John Carter,6.1,2124


In [6]:
df.shape

(4803, 10)

### Remake Columns

Some columns are useful for recommendation but arrive in a format that does not match simple rule predicates.

The following columns are transformed:

- `genres` - replace JSON id/name objects with a list of genre names only.
- `release_date` - parse to calendar year in `release_year`, then drop `release_date`; exact release day is omitted as unnecessary for basic rules.

In [7]:
df["genres"] = df["genres"].map(
    lambda v: [g["name"] for g in json.loads(str(v).strip())]
    if pd.notna(v) and str(v).strip()
    else []
)

In [8]:
df["release_year"] = (
    pd.to_datetime(df["release_date"], errors="coerce").dt.year.astype("Int64")
)
df = df.drop(columns=["release_date"])

In [9]:
df.head()

,budget,genres,original_language,popularity,revenue,runtime,title,vote_average,vote_count,release_year
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",en,150.437577,2787965087,162.0,Avatar,7.2,11800,2009
1,300000000,"[Adventure, Fantasy, Action]",en,139.082615,961000000,169.0,Pirates of the Caribbean: At World's End,6.9,4500,2007
2,245000000,"[Action, Adventure, Crime]",en,107.376788,880674609,148.0,Spectre,6.3,4466,2015
3,250000000,"[Action, Crime, Drama, Thriller]",en,112.312950,1084939099,165.0,The Dark Knight Rises,7.6,9106,2012
4,260000000,"[Action, Adventure, Science Fiction]",en,43.926995,284139100,132.0,John Carter,6.1,2124,2012


### Create Columns

Our RDR is **classification**, so the target should be a **discrete class**, not a continuous score. Predicting raw `vote_average` would be regression; binning turns the problem into ordered categories that rules can mention directly.

In [10]:
# quick range check
print("min:", df["vote_average"].min())
print("max:", df["vote_average"].max())

min: 0.0
max: 10.0


In [11]:
# create rating class with bins
df["rating_class"] = pd.cut(
    df["vote_average"],
    bins=4,
    labels=["Poor", "Average", "Good", "Excellent"],
    include_lowest=True,
    right=False
)

In [12]:
# check created bins
bins = pd.cut(
    df["vote_average"],
    bins=4,
    include_lowest=True,
    right=False
)

print(bins.cat.categories)

IntervalIndex([[0.0, 2.5), [2.5, 5.0), [5.0, 7.5), [7.5, 10.01)], dtype='interval[float64, left]')


We can see the bins are created equally as expected:

|    `vote_average`   | `rating_class` |
|---------------------|----------------|
| x &lt; 2.5          | Poor           |
| 2.5 &le; x &lt; 5.0 | Average        |
| 5.0 &le; x &lt; 7.5 | Good           |
| x &ge; 7.5          | Excellent      |

In [13]:
df.head()

,budget,genres,original_language,popularity,revenue,runtime,title,vote_average,vote_count,release_year,rating_class
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",en,150.437577,2787965087,162.0,Avatar,7.2,11800,2009,Good
1,300000000,"[Adventure, Fantasy, Action]",en,139.082615,961000000,169.0,Pirates of the Caribbean: At World's End,6.9,4500,2007,Good
2,245000000,"[Action, Adventure, Crime]",en,107.376788,880674609,148.0,Spectre,6.3,4466,2015,Good
3,250000000,"[Action, Crime, Drama, Thriller]",en,112.312950,1084939099,165.0,The Dark Knight Rises,7.6,9106,2012,Excellent
4,260000000,"[Action, Adventure, Science Fiction]",en,43.926995,284139100,132.0,John Carter,6.1,2124,2012,Good


## Preprocess Data
### Deal with Anomalies

The dataset contains non-NaN but infeasible values in several columns. These values are converted to NaN.

In [14]:
# -- CHANGE HERE --
min_feasible_budget = 1000000
min_feasible_revenue = 1000000
min_feasible_runtime = 40

In [15]:
# flag anomalies
mask_low_budget = df["budget"].apply(lambda x: x < min_feasible_budget)
mask_low_revenue = df["revenue"].apply(lambda x: x < min_feasible_revenue)
mask_low_runtime = df["runtime"].apply(lambda x: x < min_feasible_runtime)

In [16]:
# convert anomalous values to NaN
df.loc[mask_low_budget, "budget"] = np.nan
df.loc[mask_low_revenue, "revenue"] = np.nan
df.loc[mask_low_runtime, "runtime"] = np.nan

### Deal with NaNs

Both original and newly created NaNs are filled via imputation rather than dropping rows.

In [17]:
df.isnull().sum()

budget               1225
genres                  0
original_language       0
popularity              0
revenue              1619
runtime                39
title                   0
vote_average            0
vote_count              0
release_year            1
rating_class            0
dtype: int64

In [18]:
# get columns with NaNs
nan_cols = df.isnull().sum() > 0
nan_cols = nan_cols[nan_cols].index.tolist()
print(f"Columns with NaNs: {nan_cols}")
print()

# make sure they are numeric
df[nan_cols] = df[nan_cols].apply(pd.to_numeric, errors='coerce')

Columns with NaNs: ['budget', 'revenue', 'runtime', 'release_year']



In [19]:
# explore unique values (sorted by value) for columns with NaNs
for nan_col in nan_cols:
    print(f"\nFirst 10 ordered values of {nan_col}:")
    print(df[nan_col].dropna().sort_values().unique()[:10])
print()


First 10 ordered values of budget:
[1000000. 1100000. 1200000. 1250000. 1288000. 1300000. 1344000. 1350000.
 1377800. 1400000.]

First 10 ordered values of revenue:
[1000000. 1001437. 1046166. 1053788. 1072602. 1083683. 1100000. 1104682.
 1151330. 1162014.]

First 10 ordered values of runtime:
[41. 42. 46. 47. 53. 59. 60. 63. 64. 66.]

First 10 ordered values of release_year:
<IntegerArray>
[1916, 1925, 1927, 1929, 1930, 1932, 1933, 1934, 1935, 1936]
Length: 10, dtype: Int64



Imputation is performed as follows:

1. Apply group-wise imputation by `rating_class`,
2. Use a global fallback when a group statistic is unavailable.

Central tendency metrics are chosen as follows:

- `budget`: continuous and scale-sensitive $ \rightarrow $ **mean** preserves average spending level by class.
- `revenue`: continuous and scale-sensitive $ \rightarrow $ **mean** preserves average earning level by class.
- `runtime`: numeric and can be skewed $ \rightarrow $ **median** is robust to outliers.
- `release_year`: discrete and multimodal $ \rightarrow $ **mode** preserves the most plausible year category.

In [20]:
# budget imputation uses mean
budget_mean_rating_cls = df.groupby("rating_class", observed=True)["budget"].transform("mean")
budget_mean_global = df["budget"].mean()

# revenue imputation uses mean
revenue_mean_rating_cls = df.groupby("rating_class", observed=True)["revenue"].transform("mean")
revenue_mean_global = df["revenue"].mean()

# runtime imputation uses median 
runtime_median_rating_cls = df.groupby("rating_class", observed=True)["runtime"].transform("median")
runtime_median_global = df["runtime"].median()

# release_year imputation uses mode
release_mode_rating_cls = df.groupby("rating_class", observed=True)["release_year"].transform(lambda s: s.mode().iloc[0])
release_mode_global = df["release_year"].mode().iloc[0]

In [21]:
# imputation values
print(f"\nMean of budget:\n{budget_mean_rating_cls.iloc[0:3]}")
print(f"\nMean of revenue:\n{revenue_mean_rating_cls.iloc[0:3]}")
print(f"\nMedian of runtime:\n{runtime_median_rating_cls.iloc[0:3]}")
print(f"\nMode of release_year:\n{release_mode_rating_cls.iloc[0:3]}")


Mean of budget:
0    4.009226e+07
1    4.009226e+07
2    4.009226e+07
Name: budget, dtype: float64

Mean of revenue:
0    1.215670e+08
1    1.215670e+08
2    1.215670e+08
Name: revenue, dtype: float64

Median of runtime:
0    105.0
1    105.0
2    105.0
Name: runtime, dtype: float64

Mode of release_year:
0    2009
1    2009
2    2009
Name: release_year, dtype: int64


In [22]:
# impute or fill NaNs
df["budget"] = df["budget"].fillna(budget_mean_rating_cls).fillna(budget_mean_global)
df["revenue"] = df["revenue"].fillna(revenue_mean_rating_cls).fillna(revenue_mean_global)
df["runtime"] = df["runtime"].fillna(runtime_median_rating_cls).fillna(runtime_median_global)
df["release_year"] = df["release_year"].fillna(release_mode_rating_cls).fillna(release_mode_global)

In [23]:
# remove float precision
for nan_col in nan_cols:
    df[nan_col] = df[nan_col].round(0)

### Verify Post-Preprocessing

In [24]:
# verify NaNs are filled
df.isnull().sum()

budget               0
genres               0
original_language    0
popularity           0
revenue              0
runtime              0
title                0
vote_average         0
vote_count           0
release_year         0
rating_class         0
dtype: int64

In [25]:
df.head()

,budget,genres,original_language,popularity,revenue,runtime,title,vote_average,vote_count,release_year,rating_class
0,237000000.0,"[Action, Adventure, Fantasy, Science Fiction]",en,150.437577,2.787965e+09,162.0,Avatar,7.2,11800,2009,Good
1,300000000.0,"[Adventure, Fantasy, Action]",en,139.082615,9.610000e+08,169.0,Pirates of the Caribbean: At World's End,6.9,4500,2007,Good
2,245000000.0,"[Action, Adventure, Crime]",en,107.376788,8.806746e+08,148.0,Spectre,6.3,4466,2015,Good
3,250000000.0,"[Action, Crime, Drama, Thriller]",en,112.312950,1.084939e+09,165.0,The Dark Knight Rises,7.6,9106,2012,Excellent
4,260000000.0,"[Action, Adventure, Science Fiction]",en,43.926995,2.841391e+08,132.0,John Carter,6.1,2124,2012,Good


In [26]:
df.shape

(4803, 11)

## Save Data

A smaller dataset is also created to limit rule growth (reduce rule over-fragmentation) and keep the RDR tree manageable.

In [27]:
# take random 50 samples
df_subset = df.sample(n=50, random_state=42).reset_index(drop=True)
df_subset.head()

,budget,genres,original_language,popularity,revenue,runtime,title,vote_average,vote_count,release_year,rating_class
0,70000000.0,"[Action, Adventure, Comedy, Thriller]",en,13.267631,33561137.0,97.0,I Spy,5.2,269,2002,Good
1,40092264.0,"[Thriller, Action, Horror, Science Fiction, Cr...",en,4.857028,121566957.0,90.0,Split Second,5.7,63,1992,Good
2,14000000.0,"[Drama, Mystery, Thriller]",en,5.833687,5108820.0,90.0,Gossip,5.5,68,2000,Good
3,15000000.0,"[Drama, Romance]",en,32.758254,96408652.0,96.0,Vicky Cristina Barcelona,6.7,1020,2008,Good
4,250000000.0,"[Adventure, Fantasy, Family]",en,98.885637,933959197.0,153.0,Harry Potter and the Half-Blood Prince,7.4,5293,2009,Good


In [28]:
# save
df.to_csv('../data/cleaned/movies_v1.csv', index=False)
df_subset.to_csv("../data/cleaned/movies_small_v1.csv", index=False)